In [ ]:
# ODI Batting Data Analysis

This notebook loads the ODI batting dataset, cleans the columns, explores patterns in the data, and builds a simple linear regression model to predict total runs from matches played.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load dataset
path = 'odb.csv'
df = pd.read_csv(path)

# Remove the unnamed index column if present
if df.columns.str.contains('^Unnamed').any():
    df = df.loc[:, ~df.columns.str.contains('^Unnamed')]

print('Dataset shape:', df.shape)
print('\nFirst 5 rows:')
display(df.head())
print('\nColumns:', list(df.columns))
print('\nData types:\n', df.dtypes)
print('\nMissing values:\n', df.isnull().sum())


In [ ]:
# Clean dataset
# Strip whitespace from column names and convert numeric-looking columns to numeric types

df.columns = [col.strip() for col in df.columns]

# Keep the columns we need for modeling and analysis
numeric_cols = ['Mat', 'Inns', 'NO', 'Runs', 'HS', 'Ave', 'BF', 'SR', '100', '50', '0', '4s', '6s']

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Remove rows with missing values in key variables
key_cols = ['Player', 'Mat', 'Runs']
df = df.dropna(subset=[col for col in key_cols if col in df.columns]).reset_index(drop=True)

# View cleaned summary
print('Cleaned dataset shape:', df.shape)
print('\nMissing values after cleaning:\n', df.isnull().sum())
print('\nSummary statistics:\n')
display(df[numeric_cols].describe().T)


In [ ]:
# Exploratory Data Analysis (EDA)

# Correlation heatmap for numeric variables
corr = df[numeric_cols].corr(numeric_only=True)

plt.figure(figsize=(12, 8))
plt.imshow(corr, cmap='coolwarm')
plt.colorbar(label='Correlation')
plt.xticks(range(len(corr.columns)), corr.columns, rotation=45, ha='right')
plt.yticks(range(len(corr.columns)), corr.columns)
for i in range(len(corr.columns)):
    for j in range(len(corr.columns)):
        plt.text(j, i, f'{corr.iloc[i, j]:.2f}', ha='center', va='center', color='black' if abs(corr.iloc[i, j]) < 0.7 else 'white')
plt.title('Correlation Heatmap of ODI Batting Features')
plt.tight_layout()
plt.show()

# Scatter plot of Runs vs Matches
plt.figure(figsize=(8, 5))
plt.scatter(df['Mat'], df['Runs'], alpha=0.7)
plt.title('Runs vs Matches Played')
plt.xlabel('Matches Played (Mat)')
plt.ylabel('Total Runs')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Distribution of Runs
plt.figure(figsize=(8, 5))
plt.hist(df['Runs'], bins=20, color='steelblue', edgecolor='black')
plt.title('Distribution of Total Runs')
plt.xlabel('Runs')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

# Top 10 players by runs
print('\nTop 10 players by runs:\n')
print(df.sort_values('Runs', ascending=False).head(10)[['Player', 'Runs', 'Mat', 'Ave', 'SR']].to_string(index=False))


In [ ]:
# Simple Linear Regression: predict Runs from Matches
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

X = df[['Mat']]
y = df['Runs']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

predictions = model.predict(X_test)

mae = mean_absolute_error(y_test, predictions)
r2 = r2_score(y_test, predictions)
rmse = mean_squared_error(y_test, predictions, squared=False)

print('Model coefficients:')
print('Intercept:', model.intercept_)
print('Slope (Mat):', model.coef_[0])
print('\nEvaluation metrics:')
print(f'MAE: {mae:.2f}')
print(f'RMSE: {rmse:.2f}')
print(f'R^2: {r2:.4f}')

# Plot the fitted regression line
plt.figure(figsize=(8, 5))
plt.scatter(X_test, y_test, color='blue', alpha=0.6, label='Actual data')
plt.plot(X_test.sort_values('Mat'), model.predict(X_test.sort_values('Mat')), color='red', linewidth=2, label='Regression line')
plt.title('Simple Linear Regression: Runs vs Matches')
plt.xlabel('Matches Played (Mat)')
plt.ylabel('Runs')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

# Example prediction
example_matches = 150
predicted_runs = model.predict([[example_matches]])[0]
print(f'\nPredicted Runs for {example_matches} matches: {predicted_runs:.2f}')
